# Step 7: Interactive 3D Gravity Inversion Viewer
Central Sulawesi Airborne Gravity — East Sulawesi Ophiolite (ESO)

In [1]:
import os, json
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display
import discretize
import pyvista as pv
from pyproj import Transformer
from scipy.interpolate import RegularGridInterpolator

# ── Paths ─────────────────────────────────────────────────────────────────────
NB_DIR       = os.path.abspath('')   # notebooks/ dir at execution time
PROJECT_ROOT = os.path.dirname(NB_DIR)
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'output')

# ── Load outputs ──────────────────────────────────────────────────────────────
# Adaptation A: model is already in kg/m3, all cells active — load & reshape directly.
# Do NOT use actmap / InjectActiveCells.
recovered_model_kgm3 = np.load(os.path.join(OUTPUT_DIR, '07_density_model.npy'))
# NOTE: this is density CONTRAST relative to 2670 kg/m3 background (range ~ -500..+460), not absolute density

with open(os.path.join(OUTPUT_DIR, '07_mesh_params.json')) as f:
    p = json.load(f)
mesh = discretize.TensorMesh(
    [np.array(p['hx']), np.array(p['hy']), np.array(p['hz'])],
    origin=np.array(p['origin'])
)

# Reshape: mirror exactly what 07_figures.py does
assert recovered_model_kgm3.shape[0] == mesh.nC, (
    f"Model size {recovered_model_kgm3.shape[0]} != mesh.nC {mesh.nC}")
density_3d = recovered_model_kgm3.reshape(mesh.shape_cells, order='F')
# shape: (nCx, nCy, nCz)  =>  (easting, northing, z-up)

# Cell-center arrays
cc         = mesh.cell_centers
cc_east    = cc[:, 0].reshape(mesh.shape_cells, order='F')[:, 0, 0]
cc_north   = cc[:, 1].reshape(mesh.shape_cells, order='F')[0, :, 0]
cc_depth   = -(cc[:, 2].reshape(mesh.shape_cells, order='F')[0, 0, :])  # positive-down depth

# CBA grids
ds_cba  = xr.open_dataset(os.path.join(OUTPUT_DIR, 'cba_grid.nc'))
ds_pred = xr.open_dataset(os.path.join(OUTPUT_DIR, '07_predicted_cba.nc'))

northing_1d = ds_cba.northing.values
easting_1d  = ds_cba.easting.values

# Ensure northing ascending (consistent flip for both datasets)
if northing_1d[0] > northing_1d[-1]:
    northing_1d = northing_1d[::-1]
    cba_obs_2d  = ds_cba['CBA'].values[::-1, :]
    cba_pred_2d = ds_pred['CBA_predicted'].values[::-1, :]
else:
    cba_obs_2d  = ds_cba['CBA'].values
    cba_pred_2d = ds_pred['CBA_predicted'].values

misfit_2d = cba_obs_2d - cba_pred_2d

# TOPEX data
df_topex = pd.read_csv(os.path.join(PROJECT_ROOT, 'topex120124_0-3.csv'))

print(f'Mesh: {mesh.n_cells:,} cells  shape={mesh.shape_cells}')
print(f'density_3d shape: {density_3d.shape}')
print(f'Recovered model range: {recovered_model_kgm3.min():.1f} to {recovered_model_kgm3.max():.1f} kg/m3')
print(f'CBA grid shape: {cba_obs_2d.shape}  northing={northing_1d[0]:.0f}..{northing_1d[-1]:.0f}')
print(f'TOPEX rows: {len(df_topex):,}')
print('Cell 1 loaded OK')

Mesh: 152,775 cells  shape=(97, 63, 25)
density_3d shape: (97, 63, 25)
Recovered model range: -500.0 to 458.0 kg/m3
CBA grid shape: (122, 190)  northing=9762776..10005344
TOPEX rows: 43,621
Cell 1 loaded OK


## Cell 2: 3D Volume Viewer (PyVista — variable vertical mesh)

In [2]:
# Adaptation B: use mesh.to_vtk() which honours the variable hz layers.
# pyvista.ImageData with uniform spacing would misplace every layer — NOT used.
import os
os.environ['PYVISTA_OFF_SCREEN'] = 'true'
pv.OFF_SCREEN = True
pv.set_jupyter_backend('static')

# Build PyVista RectilinearGrid via discretize bridge (honors variable hz)
grid = mesh.to_vtk({'density_kgm3': recovered_model_kgm3})
print(f'Grid type: {type(grid).__name__}  n_cells: {grid.n_cells:,}')
print(f'Grid bounds: {grid.bounds}')

# Threshold to show positive-density body (likely ESO ophiolite)
clipped = grid.threshold(value=50, scalars='density_kgm3')
print(f'Clipped cells (density > 50 kg/m3): {clipped.n_cells:,}')

plotter = pv.Plotter(off_screen=True)
plotter.add_mesh(clipped, cmap='hot_r', clim=[50, 600],
                 opacity=0.7, label='Density contrast > 50 kg/m3')
plotter.add_mesh(grid.outline(), color='black', line_width=1)
plotter.show_grid()
plotter.add_title('3D Density Contrast (kg/m3) — ESO positive anomaly', font_size=10)
plotter.camera_position = 'iso'

# Export to standalone HTML (requires trame_vtk installed)
html_path = os.path.join(OUTPUT_DIR, '07_3d_viewer.html')
plotter.export_html(html_path)
html_size_kb = os.path.getsize(html_path) / 1024
print(f'Exported 3D viewer: {html_path}  ({html_size_kb:.0f} KB)')
plotter.close()
print('Cell 2 complete — PyVista path: mesh.to_vtk() -> RectilinearGrid')

Grid type: RectilinearGrid  n_cells: 152,775
Grid bounds: BoundsTuple(x_min =   246157.5286876092,
            x_max =   634157.5286876092,
            y_min =  9758776.146473428,
            y_max = 10010776.146473428,
            z_min =   -25000.0,
            z_max =        2.0)
Clipped cells (density > 50 kg/m3): 96,427


Exported 3D viewer: C:\Users\USER\belajar-vibe-coding-2\output\07_3d_viewer.html  (1590 KB)
Cell 2 complete — PyVista path: mesh.to_vtk() -> RectilinearGrid


## Cell 3: Cross-Section Explorer (N-S sections)

In [3]:
# 12 evenly spaced N-S cross sections along easting axis
section_easting_indices = np.linspace(0, len(cc_east) - 1, 12, dtype=int)

# CBA interpolator on observed grid (northing x easting)
cba_interp = RegularGridInterpolator(
    (northing_1d, easting_1d), cba_obs_2d,
    method='linear', bounds_error=False, fill_value=np.nan
)

MOHO_DEPTH_KM = 25.0   # Surono & Hartono 2013

def plot_section(section_num=0):
    sec_idx  = section_easting_indices[section_num]
    sec_east = cc_east[sec_idx]
    dens_sec = density_3d[sec_idx, :, :]   # shape (nCy, nCz)

    fig, ax = plt.subplots(figsize=(12, 5))
    pm = ax.pcolormesh(cc_north / 1e3, cc_depth / 1e3, dens_sec.T,
                       cmap='RdBu_r', vmin=-200, vmax=460, shading='auto')
    plt.colorbar(pm, ax=ax, label='Density contrast (kg/m3)')
    ax.axhline(y=MOHO_DEPTH_KM, color='black', lw=1.5, ls='--',
               label=f'Moho {MOHO_DEPTH_KM:.0f} km')
    ax.invert_yaxis()
    ax.set_xlabel('Northing (km)')
    ax.set_ylabel('Depth (km)')
    ax.set_title(
        f'N-S Section {section_num + 1}/12  |  Easting = {sec_east / 1e3:.0f} km',
        fontweight='bold'
    )
    ax.legend(loc='lower right', fontsize=8)

    # CBA profile on twin y-axis
    ax2 = ax.twinx()
    cba_prof = cba_interp(np.c_[cc_north, np.full_like(cc_north, sec_east)])
    ax2.plot(cc_north / 1e3, cba_prof, 'k-', lw=1.2, alpha=0.9, label='Observed CBA')
    ax2.set_ylabel('CBA (mGal)')
    ax2.legend(loc='upper right', fontsize=8)
    plt.tight_layout()
    plt.show()

section_slider = widgets.IntSlider(
    value=0, min=0, max=11, step=1,
    description='Section:', continuous_update=False
)
out = widgets.interactive(plot_section, section_num=section_slider)
display(out)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Section:', max=11), Output()), …

## Cell 4: Observed vs Predicted vs Misfit Map

In [4]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# UTM -> lon/lat for cartopy
transformer = Transformer.from_crs('EPSG:32751', 'EPSG:4326', always_xy=True)
east_2d_g, north_2d_g = np.meshgrid(easting_1d, northing_1d)
lon_flat, lat_flat = transformer.transform(east_2d_g.ravel(), north_2d_g.ravel())
lon_2d = lon_flat.reshape(east_2d_g.shape)
lat_2d = lat_flat.reshape(east_2d_g.shape)

show_topex = widgets.Checkbox(value=False, description='Show TOPEX overlay')

def sym_vlim(arr, pct=2):
    lim = max(abs(np.nanpercentile(arr, pct)), abs(np.nanpercentile(arr, 100 - pct)))
    return -lim, lim

def plot_maps(show_topex_val=False):
    extent = [lon_2d.min() - 0.1, lon_2d.max() + 0.1,
              lat_2d.min() - 0.1, lat_2d.max() + 0.1]
    vmin, vmax = sym_vlim(cba_obs_2d)
    mmin, mmax = sym_vlim(misfit_2d)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7),
                              subplot_kw={'projection': ccrs.PlateCarree()})
    panels = [
        (axes[0], cba_obs_2d,  'RdBu_r',  vmin, vmax, 'Observed CBA (mGal)'),
        (axes[1], cba_pred_2d, 'RdBu_r',  vmin, vmax, 'Predicted CBA (mGal)'),
        (axes[2], misfit_2d,   'coolwarm', mmin, mmax, 'Misfit (mGal)'),
    ]
    for ax, data, cmap, v0, v1, title in panels:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        pcm = ax.pcolormesh(lon_2d, lat_2d, data, cmap=cmap, vmin=v0, vmax=v1,
                            transform=ccrs.PlateCarree(), shading='auto')
        plt.colorbar(pcm, ax=ax, shrink=0.85, pad=0.02).set_label('mGal', fontsize=8)
        try:
            ax.add_feature(cfeature.NaturalEarthFeature(
                'physical', 'coastline', '10m',
                edgecolor='black', facecolor='none', linewidth=0.8))
        except Exception:
            pass
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray',
                          alpha=0.6, linestyle='--')
        gl.top_labels   = False
        gl.right_labels = False
        ax.set_title(title, fontsize=11, fontweight='bold')
    if show_topex_val:
        axes[0].scatter(df_topex['lon'], df_topex['lat'], s=0.5, c='yellow',
                        alpha=0.3, transform=ccrs.PlateCarree(), label='TOPEX')
    rmse = np.sqrt(np.nanmean(misfit_2d ** 2))
    fig.suptitle(
        f'Observed / Predicted / Misfit  |  RMSE = {rmse:.2f} mGal',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

out4 = widgets.interactive(plot_maps, show_topex_val=show_topex)
display(out4)

interactive(children=(Checkbox(value=False, description='Show TOPEX overlay'), Output()), _dom_classes=('widge…

## Cell 5: Misfit Statistics

In [5]:
from scipy.stats import pearsonr

valid  = ~(np.isnan(cba_obs_2d) | np.isnan(cba_pred_2d))
obs_v  = cba_obs_2d[valid].ravel()
pred_v = cba_pred_2d[valid].ravel()
mis_v  = (cba_obs_2d - cba_pred_2d)[valid].ravel()
r, _   = pearsonr(obs_v, pred_v)

stats = pd.DataFrame({
    'Metric': [
        'RMSE (mGal)', 'Mean Bias (mGal)', 'Std Residual (mGal)',
        'Pearson r',
        'Max density contrast (kg/m3)', 'Min density contrast (kg/m3)'
    ],
    'Value': [
        f'{np.sqrt(np.mean(mis_v**2)):.3f}',
        f'{np.mean(mis_v):.3f}',
        f'{np.std(mis_v):.3f}',
        f'{r:.4f}',
        f'{recovered_model_kgm3.max():.1f}',
        f'{recovered_model_kgm3.min():.1f}',
    ],
    'Reference': [
        '<5 mGal (Pastore 2016)', '', '',
        '>0.90 good fit',
        '<1500 kg/m3 (bound)', '>-500 kg/m3 (bound)'
    ]
})
print('=== Step 7 Inversion Summary ===')
display(stats.style.set_properties(**{'text-align': 'left'}))

=== Step 7 Inversion Summary ===


,Metric,Value,Reference
0,RMSE (mGal),2.644,<5 mGal (Pastore 2016)
1,Mean Bias (mGal),0.269,
2,Std Residual (mGal),2.630,
3,Pearson r,0.9961,>0.90 good fit
4,Max density contrast (kg/m3),458.0,<1500 kg/m3 (bound)
5,Min density contrast (kg/m3),-500.0,>-500 kg/m3 (bound)
